# 🦊 Kitsune NIDS: Quickstart Anomaly Detection Demo (Mirai Botnet)
**Canonical Reference:** Y. Mirsky et al., *Kitsune: An Ensemble of Autoencoders for Online Network Intrusion Detection*, **NDSS 2018** (DOI: [10.14722/ndss.2018.23204](https://doi.org/10.14722/ndss.2018.23204)).  
**Engineering Organization:** [kitsune-research](https://github.com/kitsune-research/Kitsune-py) | **Release:** `v0.3.1`

---

### Abstract & Execution Scope
This interactive notebook demonstrates unsupervised online network intrusion detection directly in your browser. 
The engine continuously ingests raw network packets, extracts 100 continuous statistical features in $\mathcal{O}(1)$ time complexity using damped exponential windows (AfterImage), and detects anomalous traffic patterns via an ensemble of lightweight autoencoders (KitNET) bounded by $\mathcal{O}(k^2)$ complexity.

## 1. Environment Bootstrap & Ingestion Setup
In a Google Colab environment, we clone the canonical repository and install dependencies in headless mode.

In [ ]:
import sys
from pathlib import Path

# Check if running on Google Colab or local environment
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    print("[*] Running in Google Colab environment. Bootstrapping repository...")
    !git clone --depth 1 https://github.com/kitsune-research/Kitsune-py.git repo
    %cd repo
    !pip install --upgrade pip
    !pip install -e .
    DATA_FILE = Path("mirai.pcap")
else:
    print("[*] Running in local environment.")
    DATA_FILE = Path("../mirai.pcap") if Path("../mirai.pcap").exists() else Path("mirai.pcap")

print(f"[+] Verification: Data file located at {DATA_FILE.resolve()} (Exists: {DATA_FILE.exists()})")

## 2. Streaming Ingestion & Real-Time Anomaly Scoring
We process 15,000 continuous packets from the bundled Mirai trace:
* **`FMgrace = 2,000`:** Incremental feature mapping (corClust discovers correlation topology and sets cluster ceiling $m \le 10$).
* **`ADgrace = 5,000`:** Baseline manifold training via online single-pass SGD (`max_iter = 1`). Packets are discarded from memory immediately.
* **Packets > 7,000:** Active Mirai botnet distributed scan and propagation phase.

In [ ]:
import time
import numpy as np
from Kitsune import Kitsune

PACKET_LIMIT = 15000
MAX_AE = 10
FM_GRACE = 2000
AD_GRACE = 5000

print(f"[*] Initializing Kitsune NIDS Engine...")
k_engine = Kitsune(
    file_path=str(DATA_FILE),
    limit=PACKET_LIMIT,
    max_autoencoder_size=MAX_AE,
    fm_grace=FM_GRACE,
    ad_grace=AD_GRACE
)

rmse_scores = []
start_time = time.perf_counter()

print("[*] Processing streaming packet queue...")
count = 0
while True:
    rmse = k_engine.proc_next_packet()
    if rmse == -1:
        break
    rmse_scores.append(rmse)
    count += 1
    if count % 2500 == 0:
        elapsed = time.perf_counter() - start_time
        fps = count / elapsed
        print(f"    -> Ingested {count:5d} packets | Current RMSE: {rmse:8.4f} | Throughput: {fps:6.1f} pkt/sec")

total_time = time.perf_counter() - start_time
print(f"\n[+] Completed: {len(rmse_scores)} packets evaluated in {total_time:.2f}s ({len(rmse_scores)/total_time:.1f} pkt/s)")

## 3. Dynamic Thresholding (Analytical Log-Normal Fit)
Rather than relying on fragile empirical maximums, we model the normal reconstruction residue during the stabilized baseline window ($3,000 \le t \le 7,000$) using a parametric log-normal distribution:
$$\ln(\mathrm{RMSE}) \sim \mathcal{N}(\mu_{\ln}, \sigma_{\ln}^2) \implies \tau = \exp(\mu_{\ln} + k \cdot \sigma_{\ln})$$
Setting $k = 3.0902$ bounds the theoretical False Positive Rate to $\mathrm{FPR} \le 0.001$ (at most 1 false alarm per 1,000 benign packets).

In [ ]:
# Filter out transient initialization artifacts before stabilization
baseline_window = [s for s in rmse_scores[3000:7000] if s > 0]

log_baseline = np.log(baseline_window)
mu_log = np.mean(log_baseline)
sigma_log = np.std(log_baseline)

# Statistical threshold for FPR <= 0.001 (z-score = 3.0902)
k_risk = 3.0902
tau = np.exp(mu_log + k_risk * sigma_log)

print(f"[+] Benign Manifold Baseline:")
print(f"    - Log Mean (mu_log)       : {mu_log:.4f}")
print(f"    - Log StdDev (sigma_log)  : {sigma_log:.4f}")
print(f"    - Calibrated Cutoff (tau) : {tau:.4f}")

## 4. Forensic Trajectory Visualization
Rendering the continuous RMSE response on a semilogarithmic scale against the calibrated decision boundary $\tau$.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6), dpi=140)
plt.style.use('dark_background')

x_axis = np.arange(len(rmse_scores))
plt.plot(x_axis, rmse_scores, color='#38bdf8', lw=1.2, label='KitNET RMSE Residue', alpha=0.9)
plt.axhline(y=tau, color='#f43f5e', linestyle='--', lw=1.8, label=f'Decision Boundary $\\tau = {tau:.4f}$ (FPR $\\le 10^{{-3}}$)')

# Phase demarcations
plt.axvspan(0, FM_GRACE, color='#64748b', alpha=0.25, label='corClust Feature Mapping')
plt.axvspan(FM_GRACE, AD_GRACE, color='#eab308', alpha=0.15, label='Online SGD Calibration')
plt.axvspan(7000, len(rmse_scores), color='#ef4444', alpha=0.20, label='Mirai Botnet Infiltration')

plt.yscale('log')
plt.title('Kitsune NIDS: Online Streaming Anomaly Detection (Mirai Botnet Evaluation)', fontsize=13, pad=12)
plt.xlabel('Evaluated Packets (Continuous Datastream)', fontsize=10)
plt.ylabel('RMSE Reconstruction Error (Log Scale)', fontsize=10)
plt.grid(True, which='both', ls=':', alpha=0.25)
plt.legend(loc='upper left', framealpha=0.85)
plt.tight_layout()
plt.show()

## 5. Quantitative Verification & Forensic Separation
We quantify the signal-to-noise ratio (SNR) of the anomalous attack phase ($t \ge 7,000$) compared to the benign baseline floor.

In [ ]:
attack_scores = np.array(rmse_scores[7000:])
baseline_scores = np.array(rmse_scores[3000:7000])

detected_attacks = np.sum(attack_scores > tau)
detection_ratio = (detected_attacks / len(attack_scores)) * 100
snr_median = np.median(attack_scores) / np.median(baseline_scores)
peak_separation = np.max(attack_scores) / np.median(baseline_scores)

print(f"===============================================================")
print(f" KITSUNE NIDS: RUNTIME FORENSIC SUMMARY")
print(f"===============================================================")
print(f" Total attack packets evaluated   : {len(attack_scores):,}")
print(f" Flagged packets above threshold  : {detected_attacks:,} ({detection_ratio:.2f}%)")
print(f" Separation vs Median Baseline    : {snr_median:.2f}x")
print(f" Peak Anomaly vs Noise Floor      : {peak_separation:.2f}x")
print(f"===============================================================")